# SentryNet -- Modeling and Evaluation

In [1]:
from sentrynet.config import DATA_DIR

FEATURES_PATH = DATA_DIR / "features.parquet"
DATA_AVAILABLE = FEATURES_PATH.exists()

if not DATA_AVAILABLE:
    print(f"{FEATURES_PATH} not found. Run 02_feature_engineering.ipynb first.")

## Load engineered features

In [2]:
if DATA_AVAILABLE:
    import pandas as pd

    df = pd.read_parquet(FEATURES_PATH)

## Temporal train/test split, merchant risk encoding (train-only fit), and training

In [3]:
if DATA_AVAILABLE:
    from sentrynet.data.split import temporal_split
    from sentrynet.features.merchant_risk import MerchantRiskEncoder
    from sentrynet.modeling.train import train_model

    train_df, test_df = temporal_split(df, time_col="TransactionDT")

    risk_encoder = MerchantRiskEncoder(category_col="ProductCD").fit(train_df)
    train_df = train_df.assign(merchant_risk=risk_encoder.transform(train_df))
    test_df = test_df.assign(merchant_risk=risk_encoder.transform(test_df))

    feature_cols = ["TransactionAmt", "dist1", "dist2", "velocity_1h", "time_since_last",
                     "addr_changed", "merchant_risk",
                     "device_fingerprint_degree", "card_entity_id_degree"]
    model = train_model(train_df[feature_cols], train_df["isFraud"])

## Comparison: SMOTE vs. scale_pos_weight (evaluated for completeness, not used as primary)

In [4]:
if DATA_AVAILABLE:
    from imblearn.over_sampling import SMOTE
    from sentrynet.modeling.evaluate import pr_auc
    import xgboost as xgb

    # SMOTE requires complete data, unlike XGBoost which handles NaN natively --
    # dist1/dist2 (sparse in the raw data), time_since_last (NaN for each
    # entity's first transaction), and the graph-degree features (NaN when a
    # transaction has no identity/device signal) all legitimately contain NaN.
    # This extra imputation step is itself one practical argument for
    # scale_pos_weight over SMOTE.
    X_train_imputed = train_df[feature_cols].fillna(-999)
    X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(
        X_train_imputed, train_df["isFraud"]
    )
    smote_model = xgb.XGBClassifier(eval_metric="aucpr", random_state=42)
    smote_model.fit(X_resampled, y_resampled)
    # Score with the same -999 imputation smote_model was trained on, rather
    # than raw NaN, so missing values route through its trees consistently.
    X_test_imputed = test_df[feature_cols].fillna(-999)
    smote_scores = smote_model.predict_proba(X_test_imputed)[:, 1]
    print("PR-AUC (SMOTE):", pr_auc(test_df["isFraud"], smote_scores))
    print("PR-AUC (scale_pos_weight):", pr_auc(test_df["isFraud"], model.predict_proba(test_df[feature_cols])[:, 1]))

PR-AUC (SMOTE): 0.12690248728472012
PR-AUC (scale_pos_weight): 0.16330863936613352


## Evaluation: PR-AUC and cost-based threshold

In [5]:
if DATA_AVAILABLE:
    from sentrynet.modeling.evaluate import pr_auc, select_threshold_by_cost

    test_scores = model.predict_proba(test_df[feature_cols])[:, 1]
    print("PR-AUC:", pr_auc(test_df["isFraud"], test_scores))
    best_threshold, best_cost = select_threshold_by_cost(
        test_df["isFraud"], test_scores, cost_fp=5.0, cost_fn=100.0
    )
    print("Best threshold:", best_threshold, "cost:", best_cost)

PR-AUC: 0.16330863936613352
Best threshold: 0.51 cost: 401650.0


## Save train/test splits for the drift-detection notebook

In [6]:
if DATA_AVAILABLE:
    train_df.to_parquet(DATA_DIR / "train_split.parquet")
    test_df.to_parquet(DATA_DIR / "test_split.parquet")